In [ ]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Imbalance
from imblearn.over_sampling import SMOTE

# Save model
import pickle


In [ ]:
df = pd.read_csv(r'D:\data science\projects\machice learning\customer churn prediction\data\datasetchurn.csv')

In [ ]:
df.head()

In [ ]:
df.info()
df.describe()
df.isnull().sum()

In [ ]:
df = df.drop('customerID', axis=1)

In [ ]:
df.head(10)

In [ ]:
print(df["gender"].unique())

In [ ]:
print(df["gender"].unique())

In [ ]:
# printing the unique values in all the columns

numerical_features_list = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in df.columns:
  if col not in numerical_features_list:
    print(col, df[col].unique())
    print("-"*50)

In [ ]:
print(df.isnull().sum())

In [ ]:
df[df["TotalCharges"]==" "]

In [ ]:
df["TotalCharges"] = df["TotalCharges"].replace({" ": "0.0"})
df["TotalCharges"] = df["TotalCharges"].astype(float)

In [ ]:
df.info

In [ ]:
print(df["Churn"].value_counts())

In [ ]:
def plot_histogram(df, column_name):

  plt.figure(figsize=(5, 3))
  sns.histplot(df[column_name], kde=True)
  plt.title(f"Distribution of {column_name}")

  # calculate the mean and median values for the columns
  col_mean = df[column_name].mean()
  col_median = df[column_name].median()

  # add vertical lines for mean and median
  plt.axvline(col_mean, color="red", linestyle="--", label="Mean")
  plt.axvline(col_median, color="green", linestyle="-", label="Median")

  plt.legend()

  plt.show()

In [ ]:
plot_histogram(df, "MonthlyCharges")
plot_histogram(df, "tenure")
plot_histogram(df, "TotalCharges")


In [ ]:

sns.countplot(x='Churn', data=df)
plt.title("Churn Distribution")
plt.show()

In [ ]:
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.show()

In [ ]:
sns.boxplot(x='Churn', y='tenure', data=df)
plt.show()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

In [ ]:
df.drop("customerID", axis=1, inplace=True)

In [ ]:
le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn'])

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
smote = SMOTE(k_neighbors=2, random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print(y_train.value_counts())

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr.predict(X_test)

print("Logistic Regression")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
y_pred_rf = rf.predict(X_test)

print("Random Forest")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
importances = rf.feature_importances_
features = pd.Series(importances, index=X.columns)

features.sort_values(ascending=False).head(10)

In [ ]:
with open("models/model.pkl", "wb") as f:
    pickle.dump(rf, f)

In [ ]:
sample = X_test.iloc[0].values.reshape(1, -1)
rf.predict(sample)